# 🍽️ Google Maps Platform + Gemini 3.5 Flash AI 음식점 분석 & 맞춤 여행 코스 추천
### — Google Maps Web Service API 전수 활용 & Gemini Flash 기반 지능형 맛집·카페·여행 코스 생성 시스템

본 노트북은 **Google Maps Platform API**(Places New, Geocoding, Directions 등)와 **Google Gemini 3.5 Flash LLM**을 결합하여, 특정 음식점을 기준으로 아래의 5가지 실무 시나리오를 자동으로 분석하고 맞춤형 여행 코스를 도출합니다.

---

### 🗺️ 5단계 지능형 분석 파이프라인
```
┌────────────────────────────────────────────────────────────────────────┐
│ [사용자 입력: 음식점 검색어 (예: '명동교자 본점')]                      │
└───────────────────────────────────┬────────────────────────────────────┘
                                    │
                                    ▼
┌────────────────────────────────────────────────────────────────────────┐
│ 1️⃣ 음식점 기본 정보 & 편의시설 카드 (Places API New Details `*`)       │
│    • 도로명 주소 & 우편번호, 영업시간/브레이크타임, 전화번호           │
│    • 유아의자, 화장실, 단체석, 주차, 예약가능, 포장, 배달              │
└───────────────────────────────────┬────────────────────────────────────┘
                                    │
                                    ▼
┌────────────────────────────────────────────────────────────────────────┐
│ 2️⃣ 리뷰 기반 인기 메뉴 분석 (Places Reviews ➡️ Gemini 3.5 Flash)      │
│    • 실제 방문자 리뷰 원문/번역본 + 에디토리얼 요약 수집               │
│    • Gemini AI가 대표 시그니처 메뉴, 맛의 특징, 추천 조합 분석         │
└───────────────────────────────────┬────────────────────────────────────┘
                                    │
                                    ▼
┌────────────────────────────────────────────────────────────────────────┐
│ 3️⃣ 비슷한 맛집 추천 (Places SearchNearby ➡️ Gemini 3.5 Flash)         │
│    • 동일 카테고리/가격대 반경 2km 내 맛집 탐색                         │
│    • Gemini AI가 메뉴/분위기 비교 및 추천 이유 요약                    │
└───────────────────────────────────┬────────────────────────────────────┘
                                    │
                                    ▼
┌────────────────────────────────────────────────────────────────────────┐
│ 4️⃣ 식사 후 추천 근처 카페 (Places SearchNearby ➡️ Gemini 3.5 Flash)   │
│    • 도보 5~10분(500m~800m) 내 평점 4.0+ 카페/베이커리 탐색            │
│    • 실시간 도보 거리 및 식후 입가심 디저트/커피 큐레이션              │
└───────────────────────────────────┬────────────────────────────────────┘
                                    │
                                    ▼
┌────────────────────────────────────────────────────────────────────────┐
│ 5️⃣ 맞춤형 여행 코스 생성 (Places + Directions API + Gemini Flash)     │
│    🚗 차량 여행 코스: 15km 내 드라이브 뷰포인트, 주차 여부, 소요시간   │
│    🚶 도보 여행 코스: 20분 내 힐링 산책로/문화거리, 턴바이턴 보행 가이드 │
└────────────────────────────────────────────────────────────────────────┘
```

---


## 📦 0. 환경 설정 및 API 키 로드

`.env` 파일에 설정된 `GOOGLE_MAPS_API_KEY`와 `GEMINI_API_KEY`를 자동으로 로드하고 클라이언트를 초기화합니다.


In [2]:
import os
import json
import requests
import pandas as pd
from dotenv import load_dotenv, find_dotenv
import googlemaps
from google import genai
from google.genai import types

# .env 자동 탐색 및 로드
load_dotenv(find_dotenv(), override=True)

MAPS_API_KEY = os.getenv("GOOGLE_MAPS_API_KEY", "").strip().strip('"').strip("'")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "").strip().strip('"').strip("'")

# 1. Google Maps SDK 초기화
if not MAPS_API_KEY:
    import getpass
    MAPS_API_KEY = getpass.getpass("GOOGLE_MAPS_API_KEY를 입력하세요: ").strip()

gmaps = googlemaps.Client(key=MAPS_API_KEY)
masked_maps_key = f"{MAPS_API_KEY[:6]}...{MAPS_API_KEY[-4:]}" if len(MAPS_API_KEY) > 10 else "***"
print(f"✅ Google Maps 클라이언트 초기화 완료 (키: {masked_maps_key})")

# 2. Gemini 3.5 Flash 클라이언트 초기화
if not GEMINI_API_KEY:
    import getpass
    GEMINI_API_KEY = getpass.getpass("GEMINI_API_KEY를 입력하세요: ").strip()

ai_client = genai.Client(api_key=GEMINI_API_KEY)
GEMINI_MODEL = "gemini-3.5-flash"
print(f"✅ Gemini 클라이언트 초기화 완료 (모델: {GEMINI_MODEL})")


✅ Google Maps 클라이언트 초기화 완료 (키: AIzaSy...wVj0)
✅ Gemini 클라이언트 초기화 완료 (모델: gemini-3.5-flash)


## 🎯 분석 대상 음식점 설정
원하는 음식점 이름을 아래 변수에 입력하면 전체 분석 및 코스가 자동으로 생성됩니다.


In [3]:
# 분석하고자 하는 음식점 상호명 또는 주소
TARGET_RESTAURANT = "명동교자 본점"
print(f"🎯 분석 대상 음식점: '{TARGET_RESTAURANT}'")


🎯 분석 대상 음식점: '명동교자 본점'


## 📋 1. 음식점 기본 정보 & 편의시설 조회
- **주소**: 도로명 주소, 우편번호, 좌표
- **영업시간 & 브레이크타임**: 요일별 운영 시간 및 쉬는 시간
- **전화번호**: 대표 연락처
- **부대시설**: 유아의자, 화장실, 단체석, 주차, 예약가능, 포장, 배달, 결제수단


In [4]:
# 1.1 Places API Text Search: 음식점 검색 및 Place ID 추출
search_url = "https://places.googleapis.com/v1/places:searchText"
search_headers = {
    "Content-Type": "application/json",
    "X-Goog-Api-Key": MAPS_API_KEY,
    "X-Goog-FieldMask": "places.id,places.displayName,places.formattedAddress,places.location,places.primaryType"
}
search_body = {
    "textQuery": TARGET_RESTAURANT,
    "languageCode": "ko",
    "regionCode": "kr"
}

search_res = requests.post(search_url, headers=search_headers, json=search_body).json()
places_found = search_res.get("places", [])

if not places_found:
    raise ValueError(f"'{TARGET_RESTAURANT}'을(를) 찾을 수 없습니다. 검색어를 확인해주세요.")

target_place = places_found[0]
target_place_id = target_place["id"]
target_lat = target_place["location"]["latitude"]
target_lng = target_place["location"]["longitude"]
target_coords = (target_lat, target_lng)

print(f"✅ 음식점 발견: {target_place.get('displayName', {}).get('text')} (Place ID: {target_place_id})")
print(f"📍 좌표: lat={target_lat}, lng={target_lng}")

# 1.2 Places API Details: 와일드카드 '*'로 50+ 전체 상세 속성 조회
details_url = f"https://places.googleapis.com/v1/places/{target_place_id}"
details_headers = {
    "Content-Type": "application/json",
    "X-Goog-Api-Key": MAPS_API_KEY,
    "X-Goog-FieldMask": "*"
}
details_data = requests.get(details_url, headers=details_headers, params={"languageCode": "ko"}).json()

# 1.3 기본 정보 구조화
basic_info = {
    "상호명": details_data.get("displayName", {}).get("text"),
    "대표 카테고리": details_data.get("primaryType", "음식점"),
    "표준 도로명 주소": details_data.get("formattedAddress"),
    "전화번호 (국번)": details_data.get("nationalPhoneNumber", "제공안됨"),
    "국제 전화번호": details_data.get("internationalPhoneNumber", "제공안됨"),
    "웹사이트": details_data.get("websiteUri", "없음"),
    "Google 지도 링크": details_data.get("googleMapsUri"),
    "전체 평점": f"⭐ {details_data.get('rating', 'N/A')} / 5.0 (리뷰 {details_data.get('userRatingCount', 0):,}개)"
}

print("📌 [1. 기본 정보 요약]")
for k, v in basic_info.items():
    print(f"  • {k}: {v}")

# 1.4 영업시간 및 브레이크타임
opening_hours = details_data.get("regularOpeningHours", {})
weekday_descriptions = opening_hours.get("weekdayDescriptions", [])
print("\n⏰ [2. 요일별 영업시간 & 브레이크타임]")
if weekday_descriptions:
    for desc in weekday_descriptions:
        print(f"  • {desc}")
else:
    print("  • 영업시간 정보가 등록되어 있지 않습니다.")

# 1.5 편의 및 부대시설 테이블
amenities = {
    "유아의자 / 어린이 메뉴": "✅ 제공" if details_data.get("menuForChildren") or details_data.get("goodForChildren") else ("❌ 미제공" if details_data.get("goodForChildren") is False else "정보없음"),
    "화장실 구비": "✅ 구비" if details_data.get("restroom") else "정보없음",
    "휠체어 접근 가능 화장실": "✅ 가능" if details_data.get("accessibilityOptions", {}).get("wheelchairAccessibleRestroom") else "확인필요",
    "휠체어 출입구": "✅ 완비" if details_data.get("accessibilityOptions", {}).get("wheelchairAccessibleEntrance") else "확인필요",
    "단체 이용 가능 (단체의석)": "✅ 가능" if details_data.get("goodForGroups") else ("❌ 불가" if details_data.get("goodForGroups") is False else "정보없음"),
    "야외 좌석 (테라스)": "✅ 완비" if details_data.get("outdoorSeating") else ("❌ 없음" if details_data.get("outdoorSeating") is False else "정보없음"),
    "주차 시설": "🅿️ 유료/무료 주차 제공" if any(details_data.get("parkingOptions", {}).values()) else "❌ 전용 주차장 없음 (인근 유료주차 권장)",
    "예약 가능 여부": "✅ 예약 가능" if details_data.get("reservable") else ("❌ 현장 대기" if details_data.get("reservable") is False else "확인필요"),
    "포장 (Takeout)": "✅ 가능" if details_data.get("takeout") else "정보없음",
    "배달 (Delivery)": "✅ 가능" if details_data.get("delivery") else "정보없음",
    "반려동물 동반": "🐾 가능" if details_data.get("allowsDogs") else ("❌ 불가" if details_data.get("allowsDogs") is False else "정보없음")
}

df_amenities = pd.DataFrame(list(amenities.items()), columns=["시설 및 서비스 항목", "제공 여부"])
display(df_amenities)


✅ 음식점 발견: 명동교자 본점 (Place ID: ChIJW7FBDfCifDURHTpisLbVUH0)
📍 좌표: lat=37.5610151, lng=126.9860829
📌 [1. 기본 정보 요약]
  • 상호명: 명동교자 본점
  • 대표 카테고리: dumpling_restaurant
  • 표준 도로명 주소: 대한민국 서울특별시 중구 퇴계로 129
  • 전화번호 (국번): 02-776-5348
  • 국제 전화번호: +82 2-776-5348
  • 웹사이트: http://www.mdkj.co.kr/
  • Google 지도 링크: https://maps.google.com/?cid=9029952233497836061&g_mp=CiVnb29nbGUubWFwcy5wbGFjZXMudjEuUGxhY2VzLkdldFBsYWNlEAIYBCAA
  • 전체 평점: ⭐ 4.2 / 5.0 (리뷰 14,724개)

⏰ [2. 요일별 영업시간 & 브레이크타임]
  • 월요일: 오전 10:30 ~ 오후 9:00
  • 화요일: 오전 10:30 ~ 오후 9:00
  • 수요일: 오전 10:30 ~ 오후 9:00
  • 목요일: 오전 10:30 ~ 오후 9:00
  • 금요일: 오전 10:30 ~ 오후 9:00
  • 토요일: 오전 10:30 ~ 오후 9:00
  • 일요일: 오전 10:30 ~ 오후 9:00


,시설 및 서비스 항목,제공 여부
0,유아의자 / 어린이 메뉴,정보없음
1,화장실 구비,✅ 구비
2,휠체어 접근 가능 화장실,확인필요
3,휠체어 출입구,✅ 완비
4,단체 이용 가능 (단체의석),✅ 가능
5,야외 좌석 (테라스),❌ 없음
6,주차 시설,❌ 전용 주차장 없음 (인근 유료주차 권장)
7,예약 가능 여부,❌ 현장 대기
8,포장 (Takeout),✅ 가능
9,배달 (Delivery),정보없음


## 🍜 2. 메뉴 정보 (구글 맵 리뷰 기반 인기 메뉴 분석)
Places API에서 수집된 **실제 방문자 리뷰(원문/한국어 번역)**와 **에디토리얼 요약(`editorialSummary`)**을 **Gemini 3.5 Flash** 모델에 전달하여 고객들이 가장 많이 찾는 대표 인기 메뉴와 맛의 특징을 분석합니다.


In [5]:
# 2.1 리뷰 데이터 및 에디토리얼 요약 수집
reviews_data = details_data.get("reviews", [])
editorial_summary = details_data.get("editorialSummary", {}).get("text", "에디토리얼 요약 정보 없음")

reviews_text_list = []
for idx, r in enumerate(reviews_data):
    author = r.get("authorAttribution", {}).get("displayName", "익명")
    rating = r.get("rating", 5)
    text = r.get("text", {}).get("text", "")
    orig_text = r.get("originalText", {}).get("text", text)
    rel_time = r.get("relativePublishTimeDescription", "")
    reviews_text_list.append(f"[리뷰 {idx+1}] 작성자: {author} (⭐{rating}점, {rel_time})\n- 내용: {text}\n- 원문: {orig_text}")

combined_reviews_text = "\n\n".join(reviews_text_list)

# 2.2 Gemini 3.5 Flash 프롬프트 작성
menu_prompt = f"""
당신은 대한민국 최고의 미식 전문 AI 큐레이터입니다.
아래는 Google Maps Platform에서 수집한 '{TARGET_RESTAURANT}'의 실제 고객 리뷰 및 소개 요약 데이터입니다.

[Google 에디토리얼 요약]
{editorial_summary}

[실제 고객 방문 리뷰]
{combined_reviews_text}

위 데이터를 정밀 분석하여 다음 내용을 마크다운으로 명확하게 정리해주세요:
1. 🔥 **사람들이 가장 즐겨 찾는 대표 인기 메뉴 Top 3~4** (메뉴명, 특징, 고객들의 추천 이유)
2. 👅 **맛과 식감의 핵심 포인트** (육수의 풍미, 면발, 곁들임 김치/반찬의 조화, 양과 맵기 등)
3. 💡 **첫 방문자를 위한 꿀조합 및 팁** (선불/주문 방식, 사리/밥 추가, 방문 팁 등)
"""

print("🤖 Gemini 3.5 Flash 모델로 고객 리뷰 분석 중...\n")
response_menu = ai_client.models.generate_content(
    model=GEMINI_MODEL,
    contents=menu_prompt
)

from IPython.display import Markdown
display(Markdown(response_menu.text))


Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


🤖 Gemini 3.5 Flash 모델로 고객 리뷰 분석 중...



안녕하세요. 대한민국 최고의 미식 전문 AI 큐레이터입니다. 

반세기가 넘는 시간 동안 명동을 지키며 대중의 사랑을 받아온 서울의 대표 노포이자 미쉐린 가이드 빕 구르망에 빛나는 **‘명동교자 본점’**의 정밀 분석 결과를 소개해 드립니다. 실제 방문객들의 생생한 목소리를 바탕으로, 이곳의 매력과 미식 포인트를 세밀하게 정리했습니다.

---

### 🔥 1. 사람들이 가장 즐겨 찾는 대표 인기 메뉴 Top 3

명동교자는 단 4가지의 명료한 메뉴 구성으로 승부하며, 그중에서도 독보적인 사랑을 받는 3대 메뉴가 있습니다.

*   **① 명동칼국수 (대표 시그니처)**
    *   **특징:** 일반적인 멸치나 바지락 칼국수와 달리, **닭 육수를 베이스로 하여 깊고 진한 맛**을 냅니다. 고명으로 볶은 양파와 간 고기가 올라가 고소하면서도 은은한 **'불 맛'**이 감도는 짙은 갈색의 육수가 특징입니다. 얇은 피의 물만두(변씨만두) 4개가 고명으로 함께 제공됩니다.
    *   **추천 이유:** "속이 풀리는 깊은 감칠맛", "추억을 부르는 진한 육수"라는 극찬을 받으며, 호불호 없이 깊은 만족감을 선사하는 부동의 1위 메뉴입니다.
*   **② 만두**
    *   **특징:** 얇고 투명한 만두피 속에 **부추와 돼지고기를 꽉 채워 육즙이 가득한** 정통 이북식/중화풍 만두입니다. 
    *   **추천 이유:** 한 입 베어 물면 터지는 풍부한 육즙이 일품입니다. 그대로 간장에 찍어 먹어도 훌륭하지만, 칼국수 국물에 적셔 먹을 때 그 진가가 드러납니다. 양이 푸짐하여 두 명이 방문했을 때 칼국수와 함께 곁들이기 가장 좋은 최고의 사이드 메뉴입니다.
*   **③ 비빔국수**
    *   **특징:** **클로렐라를 넣어 만든 초록색 면발**이 시각적인 즐거움을 주며, 새콤달콤하면서도 마늘의 알싸한 매운맛이 감도는 특제 양념장으로 맛을 냈습니다.
    *   **추천 이유:** 칼국수의 뜨겁고 진한 맛과 대비되는 매콤하고 개운한 맛으로, 입맛을 돋우는 훌륭한 파트너 역할을 합니다.

---

### 👅 2. 맛과 식감의 핵심 포인트

*   **육수의 풍미: '묵직함과 불 맛의 조화'**
    *   닭 육수 특유의 담백함과 고기 고명, 볶은 양파에서 우러나오는 감칠맛이 더해져 매우 묵직하고 진합니다. 마치 잘 우려낸 소갈비탕이나 중화풍 탕면을 연상시키는 깊은 불 맛이 감돌아 해장용으로도 탁월합니다. 다만, 맑고 깔끔한 한국식 칼국수를 선호하는 분들에게는 다소 무겁거나 느끼하게 느껴질 수 있습니다.
*   **면발: '부드럽고 매끄러운 목 넘김'**
    *   명동교자의 면발은 쫄깃한 탄성보다는 **부드럽고 하늘하늘한 식감**이 특징입니다. 국물을 가득 머금어 후루룩 부드럽게 넘어가며, 소화가 잘되는 편안한 식감을 자랑합니다. (※ 꼬들꼬들하고 쫄깃한 면을 선호하시는 분들에게는 다소 퍼진 느낌으로 다가올 수 있습니다.)
*   **곁들임 김치: '중독성 있는 강렬한 마늘 김치'**
    *   명동교자를 이야기할 때 빼놓을 수 없는 핵심 조연입니다. 알싸한 마늘을 아낌없이 넣어 **혀끝이 아릴 정도로 강한 매운맛과 톡 쏘는 향**을 자랑합니다. 이 강렬한 김치가 칼국수의 묵직한 기름진 맛을 완벽하게 잡아주어 환상의 궁합을 이룹니다. 매운맛에 약하거나 마늘 향을 꺼리는 분들에게는 다소 자극적일 수 있으나, 한 번 빠지면 헤어나올 수 없는 마성의 매력을 가졌습니다.

---

### 💡 3. 첫 방문자를 위한 꿀조합 및 이용 팁

*   **초고속 시스템 (주문 및 결제):**
    *   이곳은 **'선불 결제 시스템'**입니다. 테이블 안내와 동시에 주문과 결제가 이루어지며, 자리에 앉자마자 음식이 5분 내로 서빙되는 엄청난 회전율을 자랑합니다. 미리 메뉴를 결정하고 카드를 준비해 두는 것이 좋습니다.
*   **프로 미식가의 무료 리필 혜택:**
    *   명동교자는 인원수대로 주문 시 **'면 사리'와 '공깃밥(조 조그만 밥)'을 무료로 리필**해 줍니다. 국수를 어느 정도 드신 후, 남은 진한 고기 육수에 밥을 말아 마늘 김치를 얹어 먹는 '국밥 스타일 마무리'는 단골들이 가장 사랑하는 코스입니다.
*   **대기 시간 및 회전율:**
    *   늘 대기 줄이 길게 늘어서 있지만, 단일 메뉴 구성과 빠른 서빙 덕분에 **회전율이 매우 빠릅니다.** 보통 5~10분 내외로 입장이 가능하니 줄이 길더라도 포기하지 마세요.
*   **식후 에티켓 팁:**
    *   김치의 마늘 향이 매우 강해 식사 후 오랫동안 입안에 마늘 잔향이 남습니다. 제공되는 자일리톨 껌을 꼭 챙기시고, 중요한 미팅이나 데이트 직전이라면 이 점을 염두에 두시는 것이 좋습니다.

## 🍲 3. 이 음식점과 비슷한 유사 맛집 추천
음식점의 카테고리(`primaryType`), 평점, 가격대, 지리적 위치를 기반으로 반경 2km 이내 유사 맛집을 탐색한 뒤, **Gemini 3.5 Flash**가 맛과 분위기를 대조하여 추천 사유를 제시합니다.


In [6]:
# 3.1 반경 2km 이내 동일/유사 카테고리 음식점 검색
nearby_res_url = "https://places.googleapis.com/v1/places:searchNearby"
nearby_res_headers = {
    "Content-Type": "application/json",
    "X-Goog-Api-Key": MAPS_API_KEY,
    "X-Goog-FieldMask": "places.id,places.displayName,places.formattedAddress,places.rating,places.userRatingCount,places.primaryType,places.location"
}
nearby_res_body = {
    "includedTypes": ["korean_restaurant", "restaurant"],
    "maxResultCount": 6,
    "locationRestriction": {
        "circle": {
            "center": {"latitude": target_lat, "longitude": target_lng},
            "radius": 2000.0  # 반경 2km
        }
    },
    "languageCode": "ko"
}

similar_raw = requests.post(nearby_res_url, headers=nearby_res_headers, json=nearby_res_body).json()
similar_places = [p for p in similar_raw.get("places", []) if p.get("id") != target_place_id][:4]

similar_candidates = []
for p in similar_places:
    similar_candidates.append({
        "상호명": p.get("displayName", {}).get("text"),
        "평점": f"⭐ {p.get('rating', 'N/A')}",
        "리뷰수": p.get("userRatingCount", 0),
        "주소": p.get("formattedAddress"),
        "Place ID": p.get("id")
    })

df_similar = pd.DataFrame(similar_candidates)
print(f"✅ 반경 2km 내 유사 맛집 후보 {len(similar_candidates)}곳 발견:")
display(df_similar)

# 3.2 Gemini 3.5 Flash에게 유사 맛집 비교 추천 요청
similar_prompt = f"""
기준 맛집: '{TARGET_RESTAURANT}' (상호명: {details_data.get('displayName', {}).get('text')})
아래는 Google Maps Platform에서 검색된 인근 맛집 후보 목록입니다:
{json.dumps(similar_candidates, ensure_ascii=False, indent=2)}

위 후보들 중 '{TARGET_RESTAURANT}'을 방문하려던 미식가가 함께 고려해볼 만한 유사/대체 맛집 2~3곳을 선정하고:
1. 각 식당의 매력 및 메뉴/분위기 비교 포인트
2. 본점 대신 또는 2차 식사 장소로 방문했을 때의 장점 (웨이팅 분산, 특색 있는 요리 등)
을 친절하게 비교 추천해주세요.
"""

response_similar = ai_client.models.generate_content(
    model=GEMINI_MODEL,
    contents=similar_prompt
)
display(Markdown(response_similar.text))


✅ 반경 2km 내 유사 맛집 후보 4곳 발견:


,상호명,평점,리뷰수,주소,Place ID
0,보코서울명동,⭐ 4.2,564,대한민국 서울특별시 중구 퇴계로 52,ChIJwdSjpvWifDURKT_mblhMHIA
1,오다리집 간장게장,⭐ 4.8,7025,"2F, 28 명동8나길 중구 서울특별시 대한민국",ChIJNbhaHPGifDURDCGKakYKwXw
2,부촌육회 본점,⭐ 4.4,2215,대한민국 서울특별시 종로구 종로 200-12,ChIJQR1ODmCjfDUREpgR7iCRQDI
3,BHC치킨 명동본점,⭐ 3.8,2263,대한민국 서울특별시 중구 명동7길 21,ChIJCT9h1O-ifDURQuD7FuVo3-Q


명동의 대표적인 미식 보석인 **'명동교자 본점'**은 진한 닭 육수 칼국수와 알싸한 마늘 김치, 그리고 육즙 가득한 만두로 독보적인 사랑을 받는 곳입니다. 하지만 늘 긴 웨이팅이 있고, 회전율이 빨라 느긋하게 대화를 나누며 식사하기는 조금 어렵다는 아쉬움이 있습니다.

제시해주신 인근 후보지 중, 명동교자를 방문하려던 미식가의 취향(전통적인 맛, 대중적인 인지도, 미식 가치)을 고려하여 **함께 방문하거나 대체하기 좋은 맛집 3곳**을 선정해 비교 추천해 드립니다.

---

### 1. 오다리집 간장게장 (대체재: 명동에서 즐기는 프리미엄 한식)
* **평점:** ⭐ 4.8 (리뷰 7,025개)
* **위치:** 명동 내 (명동교자 도보 3~5분 거리)

**비교 포인트 (매력 및 분위기)**
* **메뉴 & 분위기:** 명동교자가 '칼국수와 만두'라는 서민적이고 따뜻한 탄수화물 중심의 식사라면, 오다리집은 한국의 또 다른 대표 밥도둑인 **'간장게장'**을 전문으로 합니다. 라이브 음악이 흐르는 이색적이고 활기찬 분위기로, 외국인 관광객뿐만 아니라 국내 미식가들에게도 게장의 신선함과 감칠맛으로 극찬받는 곳입니다.
* **대체 방문 시 장점:** 명동교자의 무거운 대기 줄에 지쳤을 때, **조금 더 대접받는 느낌의 정갈한 프리미엄 한식**을 원하신다면 최고의 선택입니다. 짭조름하고 고소한 게장 정식은 명동교자의 칼칼한 마늘 김치와는 또 다른 매력의 한국적인 감칠맛을 선사합니다.

---

### 2. BHC치킨 명동본점 (2차 장소: '치맥'으로 이어지는 명동 밤의 열기)
* **평점:** ⭐ 3.8 (리뷰 2,263개)
* **위치:** 명동 내 (명동교자 도보 5분 거리)

**비교 포인트 (매력 및 분위기)**
* **메뉴 & 분위기:** 명동교자는 식사 후 바로 자리를 비워줘야 하는 '빠른 회전율' 중심의 밥집인 반면, BHC치킨은 넓고 쾌적한 공간에서 시원한 맥주와 바삭한 한국식 치킨(뿌링클, 골드킹 등)을 즐길 수 있는 **캐주얼하고 활기찬 펍 분위기**입니다.
* **2차 식사 장소로서의 장점:** 명동교자에서 칼국수와 만두로 든든하게 1차 식사를 마친 후, **가볍게 술 한잔하며 이야기를 나누고 싶을 때 완벽한 2차 장소**입니다. 혹은 명동교자의 엄청난 대기 줄을 피해 넓은 매장에서 편안하게 대중적인 K-푸드(치킨)를 즐기고 싶을 때 훌륭한 대안이 됩니다.

---

### 3. 부촌육회 본점 (미식가 코스: 미쉐린 가이드의 또 다른 전설)
* **평점:** ⭐ 4.4 (리뷰 2,215개)
* **위치:** 서울 종로구 종로 200-12 (광장시장 내, 명동에서 대중교통/택시로 약 15분)

**비교 포인트 (매력 및 분위기)**
* **메뉴 & 분위기:** 명동교자와 마찬가지로 **'미쉐린 가이드 빕 구르망(Bib Gourmand)'**에 오랜 기간 선정된 검증된 맛집입니다. 신선하고 고소한 육회와 낙지탕탕이가 주메뉴이며, 활기 넘치는 광장시장 골목의 정겨운 분위기를 품고 있습니다.
* **대체/연계 방문 시 장점:** 명동교자 본점의 미식 수준에 걸맞은 **또 다른 최고 수준의 로컬 맛집을 탐방하고자 하는 미식가에게 추천**합니다. 칼국수가 다소 무겁게 느껴진다면, 신선하고 가벼운 육회와 육회비빔밥은 훌륭한 대체제가 됩니다. 명동에서 쇼핑을 즐긴 후 광장시장으로 이동해 미식 투어를 이어가는 코스로도 매우 좋습니다.

---

### 요약 추천 가이드

* **"명동 안에서 줄 서지 않고 고급스러운 한식을 먹고 싶다"** ➡️ **오다리집 간장게장**
* **"명동교자에서 빠르게 식사한 후, 맥주 한잔하며 수다를 떨고 싶다"** ➡️ **BHC치킨 명동본점**
* **"명동교자만큼 명성 있는 서울의 진짜 미쉐린 맛집을 경험하고 싶다"** ➡️ **부촌육회 본점**

## ☕ 4. 식사 후 추천 근처 카페
음식점에서 식사를 마친 후 **도보 5~10분 (반경 500m~800m)** 내에 이동할 수 있는 평점 4.0 이상의 카페 및 베이커리를 검색하고, 식후 입가심에 어울리는 최적의 카페를 추천합니다.


In [7]:
# 4.1 도보 권역(반경 700m) 내 카페 검색
cafe_url = "https://places.googleapis.com/v1/places:searchNearby"
cafe_headers = {
    "Content-Type": "application/json",
    "X-Goog-Api-Key": MAPS_API_KEY,
    "X-Goog-FieldMask": "places.id,places.displayName,places.formattedAddress,places.rating,places.userRatingCount,places.location,places.outdoorSeating"
}
cafe_body = {
    "includedTypes": ["cafe", "coffee_shop", "bakery"],
    "maxResultCount": 5,
    "locationRestriction": {
        "circle": {
            "center": {"latitude": target_lat, "longitude": target_lng},
            "radius": 700.0  # 도보 약 10분 이내
        }
    },
    "languageCode": "ko"
}

cafe_raw = requests.post(cafe_url, headers=cafe_headers, json=cafe_body).json()
cafe_places = cafe_raw.get("places", [])

# 4.2 직선거리 및 도보 시간 계산
import math
def haversine_meters(lat1, lon1, lat2, lon2):
    R = 6371000
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlambda/2)**2
    return 2 * R * math.atan2(math.sqrt(a), math.sqrt(1 - a))

cafe_list = []
for c in cafe_places:
    c_lat = c["location"]["latitude"]
    c_lng = c["location"]["longitude"]
    dist_m = haversine_meters(target_lat, target_lng, c_lat, c_lng)
    walk_min = max(1, round(dist_m / 65))  # 평균 보행속도 분당 65m
    
    cafe_list.append({
        "카페명": c.get("displayName", {}).get("text"),
        "평점": f"⭐ {c.get('rating', 'N/A')}",
        "리뷰수": c.get("userRatingCount", 0),
        "도보 거리": f"{int(dist_m)} m",
        "도보 예상 시간": f"약 {walk_min} 분",
        "야외 좌석": "✅ 있음" if c.get("outdoorSeating") else "실내 좌석",
        "주소": c.get("formattedAddress"),
        "lat": c_lat,
        "lng": c_lng
    })

df_cafes = pd.DataFrame(cafe_list)
print(f"✅ 식후 도보 이동 가능한 근처 카페 {len(cafe_list)}곳 탐색 완료:")
display(df_cafes[["카페명", "평점", "리뷰수", "도보 거리", "도보 예상 시간", "야외 좌석", "주소"]])

# 4.3 Gemini 3.5 Flash의 식후 맞춤 카페 페어링 큐레이션
cafe_prompt = f"""
식사한 음식점: '{TARGET_RESTAURANT}' (진하고 깊은 국물/마늘 김치가 특징인 음식)
식후 방문 가능한 인근 카페 목록:
{json.dumps(cafe_list, ensure_ascii=False, indent=2)}

식사를 마친 손님이 입안을 깔끔하게 정리하고 담소를 나누기에 가장 적합한 카페 2~3곳을 선정하여:
1. 식후 음료/디저트 페어링 포인트 (예: 깔끔한 산미의 드립커피, 시원한 아메리카노, 시그니처 디저트)
2. 매장 분위기 및 좌석 편의성 (조용한 대화, 채광, 테라스 등)
을 추천해주세요.
"""

response_cafe = ai_client.models.generate_content(
    model=GEMINI_MODEL,
    contents=cafe_prompt
)
display(Markdown(response_cafe.text))


✅ 식후 도보 이동 가능한 근처 카페 5곳 탐색 완료:


,카페명,평점,리뷰수,도보 거리,도보 예상 시간,야외 좌석,주소
0,Rewire coffee,⭐ 4.9,526,536 m,약 8 분,✅ 있음,대한민국 서울특별시 중구 수표로6길 24-1
1,블루보틀 명동 카페,⭐ 4.5,255,395 m,약 6 분,실내 좌석,대한민국 서울특별시 중구 명동길 14
2,맷차 명동본점,⭐ 4.5,845,420 m,약 6 분,실내 좌석,대한민국 서울특별시 중구 명동9길 17
3,명동맘하우스,⭐ 4,259,81 m,약 1 분,실내 좌석,대한민국 서울특별시 중구 남산동3가 13-21
4,커피한약방,⭐ 4.5,2388,651 m,약 10 분,✅ 있음,대한민국 서울특별시 중구 삼일대로12길 16-6


명동교자 본점의 **진하고 깊은 고기 국물**과 **알싸하고 강한 마늘 김치**를 드신 후에는 입안의 텁텁함과 마늘 향을 깔끔하게 잡아주고, 편안하게 대화를 나눌 수 있는 카페가 필요합니다. 

제시해주신 목록 중 이러한 '식후 입가심'과 '편안한 담소'에 가장 적합한 **카페 3곳**을 엄선하여 추천해 드립니다.

---

### 1. 맷차 명동본점 (도보 6분, 420m)
**"마늘 향을 완벽하게 잡아줄 맷돌 말차의 깔끔함과 넓은 공간"**

*   **식후 음료/디저트 페어링 포인트:**
    *   **추천 음료:** **맷돌 말차 오리지널(또는 말차 밀크티)** 
    *   **페어링 이유:** 녹차와 말차에 풍부하게 함유된 '카테킨' 성분은 **마늘 냄새(알리신 성분)를 제거하는 데 탁월한 효과**가 있습니다. 명동교자의 강한 마늘 맛을 지우고 입안을 가장 빠르고 개운하게 정돈해 줍니다. 단맛이 적고 쌉싸름한 맷차 고유의 음료가 기름진 국물의 맛도 씻어내 줍니다.
*   **매장 분위기 및 좌석 편의성:**
    *   총 4층 규모의 대형 매장으로, 명동 중심가에서 보기 드물게 **좌석 간격이 넓고 쾌적**합니다. 
    *   통창을 통해 들어오는 채광이 좋으며, 소란스럽지 않고 차분한 분위기 속에서 눈치 보지 않고 편안하게 장시간 대화를 나누기에 최적의 공간입니다.

---

### 2. 블루보틀 명동 카페 (도보 6분, 395m)
**"칼국수의 묵직함을 날려줄 화사한 산미의 드립 커피"**

*   **식후 음료/디저트 페어링 포인트:**
    *   **추천 음료:** **싱글 오리진 푸어오버(드립 커피) 또는 놀라(Nola)**
    *   **페어링 이유:** 명동교자의 묵직한 닭고기 육수 뒤에는 화사하고 깔끔한 산미가 있는 필터 커피가 아주 잘 어울립니다. 에스프레소 머신으로 내린 커피보다 **종이 필터로 걸러내어 오일감이 없는 푸어오버 커피**가 입안을 가볍고 산뜻하게 리셋해 줍니다.
*   **매장 분위기 및 좌석 편의성:**
    *   블루보틀 특유의 화이트&우드 톤의 미니멀하고 세련된 인테리어입니다.
    *   통유리창을 통해 명동 거리를 내려다볼 수 있는 뷰가 좋으며, 밝고 경쾌한 분위기 속에서 트렌디한 감성과 함께 가벼운 담소를 나누기 좋습니다.

---

### 3. 커피한약방 (도보 10분, 651m)
**"소화를 돕는 산책과 레트로한 감성 속 깊은 대화"**

*   **식후 음료/디저트 페어링 포인트:**
    *   **추천 음료:** **필터 커피 (을지로 블렌드 / 디카페인 가능)**
    *   **페어링 이유:** 옛날 방식으로 장인이 직접 로스팅한 원두를 손으로 내려주는 필터 커피가 주 메뉴입니다. 얼음 가득한 필터 커피 한 잔은 입안의 기름기를 싹 걷어가 줍니다. 바로 앞 디저트 전문점 '혜민당'의 달콤한 타르트를 곁들이면 단짠(단맛+짠맛)의 완벽한 마무리가 됩니다.
*   **매장 분위기 및 좌석 편의성:**
    *   **도보로 약 10분 거리**에 있어, 식후 가볍게 산책하며 명동교자의 배부름을 소화시키기에 딱 좋은 거리에 있습니다.
    *   조선시대 '혜민서' 터에 위치한 초입부터 개화기 풍의 독특하고 아늑한 분위기를 풍깁니다. 좁은 골목길 사이의 야외 좌석(테라스 느낌)이나 클래식한 음악이 흐르는 실내 공간은 일상에서 벗어난 특별한 대화 시간을 선사합니다.

---

**💡 요약 가이드**
*   **마늘 향과 입가심이 가장 시급하다면?** ➡️ **맷차** (말차의 효능)
*   **깔끔하고 화사한 커피 맛을 선호한다면?** ➡️ **블루보틀 명동**
*   **소화도 시킬 겸 특별한 분위기에서 깊은 대화를 원한다면?** ➡️ **커피한약방**

## 🚗🚶 5. 음식점 근처 맞춤 여행 코스
식사 후 즐길 수 있는 **차량 드라이브 코스**와 **도보 산책 코스**를 Google Maps Directions API 및 Gemini 3.5 Flash로 설계합니다.


In [8]:
# 5.1 주변 대표 관광 명소 탐색 (Places API)
tourist_url = "https://places.googleapis.com/v1/places:searchNearby"
tourist_headers = {
    "Content-Type": "application/json",
    "X-Goog-Api-Key": MAPS_API_KEY,
    "X-Goog-FieldMask": "places.id,places.displayName,places.formattedAddress,places.rating,places.location,places.primaryType"
}
tourist_body = {
    "includedTypes": ["tourist_attraction", "park", "historical_landmark", "museum"],
    "maxResultCount": 6,
    "locationRestriction": {
        "circle": {
            "center": {"latitude": target_lat, "longitude": target_lng},
            "radius": 3000.0  # 반경 3km
        }
    },
    "languageCode": "ko"
}

tourist_raw = requests.post(tourist_url, headers=tourist_headers, json=tourist_body).json()
attractions = []
for t in tourist_raw.get("places", []):
    attractions.append({
        "명소명": t.get("displayName", {}).get("text"),
        "평점": f"⭐ {t.get('rating', 'N/A')}",
        "주소": t.get("formattedAddress"),
        "coords": (t["location"]["latitude"], t["location"]["longitude"])
    })

df_attractions = pd.DataFrame([{"명소명": a["명소명"], "평점": a["평점"], "주소": a["주소"]} for a in attractions])
print("🏛️ [주변 주요 관광/문화 명소 목록]")
display(df_attractions)

# 5.2 Gemini 3.5 Flash를 활용한 차량 드라이브 코스 & 도보 산책 코스 종합 생성
itinerary_prompt = f"""
출발점 (식사 장소): '{TARGET_RESTAURANT}' (주소: {details_data.get('formattedAddress')})
주변 탐색된 관광 명소 및 문화 유적:
{json.dumps([a['명소명'] for a in attractions], ensure_ascii=False)}

위 정보를 바탕으로 식사 후 이어지는 완벽한 2가지 테마 여행 코스를 작성해주세요:

---
### 🚗 1. 차량 드라이브 코스 (Half-Day Driving Course)
- **추천 대상**: 드라이브, 야경, 뷰포인트 감상을 원하는 방문객
- **추천 경로**: {TARGET_RESTAURANT} ➡️ [주요 드라이브 명소 1~2곳 (예: 남산 순환로, 북악스카이웨이, 한강 뷰포인트 등)] ➡️ [일몰/야경 카페]
- **포인트**: 각 스팟별 주차 편의성, 추천 드라이브 시간대, 예상 차량 소요시간

---
### 🚶 2. 힐링 도보 산책 코스 (Pedestrian Walking Tour)
- **추천 대상**: 식사 후 가볍게 소화시키며 도심 문화를 즐기려는 도보 여행자
- **추천 경로**: {TARGET_RESTAURANT} ➡️ [도보 5~15분 거리 명소 (예: 명동성당, 청계천 산책로, 남산골 한옥마을 등)]
- **포인트**: 총 보행 시간 (약 20~40분), 포토존, 산책 힐링 포인트
"""

print("🗺️ Gemini 3.5 Flash가 맞춤형 차량/도보 여행 코스를 설계 중입니다...\n")
response_itinerary = ai_client.models.generate_content(
    model=GEMINI_MODEL,
    contents=itinerary_prompt
)
display(Markdown(response_itinerary.text))


🏛️ [주변 주요 관광/문화 명소 목록]


,명소명,평점,주소
0,경복궁,⭐ 4.6,대한민국 서울특별시 종로구 사직로 161
1,광장시장,⭐ 4.2,대한민국 서울특별시 종로구 청계천로 88
2,N서울타워,⭐ 4.5,대한민국 서울특별시 용산구 남산공원길 105
3,북촌 한옥마을,⭐ 4.4,대한민국 서울특별시 종로구 계동길
4,명동거리,⭐ 4.4,대한민국 서울특별시 중구 명동2가
5,남대문시장,⭐ 4.2,대한민국 서울특별시 중구 남대문시장4길 21


🗺️ Gemini 3.5 Flash가 맞춤형 차량/도보 여행 코스를 설계 중입니다...



맛있는 식사 후, 서울 중심부에서 즐길 수 있는 가장 완벽한 동선의 2가지 테마 여행 코스를 제안해 드립니다. 

---

### 🚗 1. 차량 드라이브 코스 (Half-Day Driving Course)
**"서울의 화려한 빌딩 숲과 자연이 한눈에, 낭만 야경 드라이브"**

명동교자에서 든든하게 식사를 마친 후, 차를 타고 남산과 북악산의 능선을 따라 서울의 아름다운 일몰과 야경을 감상하는 코스입니다.

*   **추천 경로:**
    명동교자 본점 ➡️ **남산 소월로 (드라이브 코스)** ➡️ **북악스카이웨이 팔각정 (뷰포인트)** ➡️ **성북동 '산모퉁이' 또는 '아델라베일리' (일몰/야경 카페)**

```
 [명동교자 본점] 🚗 (10분) ➡️ [남산 소월로] 🚗 (25분) ➡️ [북악팔각정] 🚗 (10분) ➡️ [전망 좋은 성북동 카페]
```

*   **코스 상세 정보:**
    1.  **남산 소월로 (드라이브):** 명동에서 출발해 숭례문을 거쳐 남산 소월로로 진입합니다. 우측으로 펼쳐지는 용산과 한강의 풍경, 좌측의 남산 타워를 품고 달리는 서울 최고의 도심 드라이브 명소입니다.
    2.  **북악스카이웨이 팔각정 (야경):** 인왕산과 북악산 자락을 따라 꼬불꼬불한 숲길을 운전해 올라가면 팔각정에 도착합니다. 서울 시내(경복궁, 롯데타워 등)와 북한산 자락이 360도로 펼쳐지는 최고의 뷰포인트입니다.
    3.  **성북동 카페 아델라베일리 or 산모퉁이:** 북악스카이웨이에서 차로 10분 거리의 성북동 골목에 위치한 카페입니다. 야외 테라스에서 지는 노을과 서울 성곽길 야경을 바라보며 따뜻한 차 한 잔을 즐기기 좋습니다.

*   **핵심 포인트:**
    *   **추천 드라이브 시간대:** 일몰 1시간 전 출발 추천 (예: 오후 5시 30분 ~ 저녁 8시 30분). 낮에서 밤으로 변하는 서울의 스카이라인을 모두 볼 수 있습니다.
    *   **예상 차량 소요시간:** 편도 순수 주행 시간 약 45~50분 (교통 상황에 따라 변동 가능).
    *   **주차 편의성:**
        *   *명동교자:* 명동 특성상 전용 주차장이 없으므로 주변 '명동 롯데백화점' 혹은 '엠플라자 주차장' 이용 권장.
        *   *북악팔각정:* 자체 공영주차장 완비 (주말 저녁에는 대기가 있을 수 있으나 회전율이 빠름).
        *   *추천 카페:* 카페 전용 주차 공간 및 발레파킹 지원으로 편리함.

---

### 🚶 2. 힐링 도보 산책 코스 (Pedestrian Walking Tour)
**"도심 속 고요함을 찾아서, 역사와 자연이 숨 쉬는 힐링 산책"**

명동교자의 마늘 향 가득한 칼국수와 만두를 맛있게 소화시키며, 복잡한 명동거리에서 벗어나 고즈넉한 분위기를 만끽할 수 있는 코스입니다.

*   **추천 경로:**
    명동교자 본점 ➡️ **명동성당 (도보 5분)** ➡️ **남산골 한옥마을 (도보 15분)**

```
 [명동교자 본점] 🚶 (도보 5분) ➡️ [명동성당] 🚶 (도보 15분) ➡️ [남산골 한옥마을]
```

*   **코스 상세 정보:**
    1.  **명동성당 (사색과 포토타임):** 명동교자에서 나와 조금만 걸어 올라가면 장엄한 고딕 양식의 명동성당이 나타납니다. 성당 내부의 고요함 속에서 마음을 차분히 가라앉히거나, 성당 뒤편의 아담한 성모 동산 산책로를 걸으며 힐링할 수 있습니다.
    2.  **남산골 한옥마을 (한국의 美와 자연):** 명동성당에서 퇴계로 방향으로 내려와 충무로역 뒤편으로 걸어가면 서울 한복판에 숨겨진 전통 한옥마을이 나옵니다. 남산 자락 아래 시냇물이 흐르고, 전통 가옥들과 현대적인 N서울타워가 한눈에 담기는 이색적이고 평화로운 공간입니다.

*   **핵심 포인트:**
    *   **총 보행 시간:** 약 20~25분 (순수 걷는 시간 기준, 총 이동 거리 약 1.3km로 경사가 완만하여 소화시키기에 최적의 코스).
    *   **포토존 추천:**
        *   *명동성당 뷰 카페 '몰또(Molto)':* 에스프레소 바 테라스에서 성당을 배경으로 이국적인 사진을 남길 수 있습니다.
        *   *남산골 한옥마을 '천우각(지우루)':* 전통 연못과 정자, 그리고 저 멀리 보이는 N서울타워를 한 프레임에 담을 수 있는 최고의 포토스팟입니다.
    *   **산책 힐링 포인트:** 남산골 한옥마을 뒤편의 '서울천년타임캡슐 광장'은 소음이 차단된 넓은 광장으로, 한적하게 하늘을 바라보며 누워 쉬거나 사색을 즐기기에 매우 훌륭합니다.

## 📊 6. 최종 종합 요약 및 활용 가이드

본 노트북은 **Google Maps Platform API**의 실시간 지리정보(장소 상세, 편의시설, 리뷰, 주변 검색)를 데이터 파이프라인으로 삼고, **Gemini 3.5 Flash**의 다국어 텍스트 이해 및 추론 역량을 결합하여 상용 서비스 수준의 **AI 맛집 & 여행 플래너**를 성공적으로 구축하였습니다.

---

### 💡 다른 지역 / 음식점으로 확장하는 방법
노트북 상단의 `TARGET_RESTAURANT` 변수를 원하는 음식점으로 변경하고 전체 셀을 다시 실행하면 모든 분석과 여행 코스가 즉시 새로 생성됩니다:
```python
TARGET_RESTAURANT = "강남파이낸스센터 인근 맛집"  # 또는 "성수동 소문난 성수 감자탕", "해운대 암소갈비집" 등
```
